# Explainability — SHAP Analysis

This notebook applies SHAP (SHapley Additive exPlanations) to the trained XGBoost model to explain both global feature importance and individual predictions. This is the core interpretability layer that lets loan officers and regulators see *why* the model flagged a given applicant as high or low risk.

Plots are saved to `../data/processed/`.

## 1. Imports

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

shap.initjs()


## 2. Load Trained XGBoost Model

In [ ]:
model = joblib.load("../models/xgb_model.pkl")
print("Loaded XGBoost model:", type(model))


## 3. Load Processed Data and Sample 500 Rows

SHAP value computation scales with dataset size, so we work with a random sample for interactive analysis.

In [ ]:
df = pd.read_csv("../data/processed/train_processed.csv")
print(f"Full dataset shape: {df.shape}")

sample_df = df.sample(n=500, random_state=42).reset_index(drop=True)
X_sample = sample_df.drop(columns=["TARGET"])
y_sample = sample_df["TARGET"]

print(f"Sample shape: {X_sample.shape}")
print(f"Sample target distribution:\n{y_sample.value_counts()}")


## 4. Create SHAP TreeExplainer

In [ ]:
explainer = shap.TreeExplainer(model)
print("SHAP TreeExplainer created for XGBoost model.")


## 5. Calculate SHAP Values

In [ ]:
shap_values = explainer(X_sample)

print(f"SHAP values shape: {shap_values.values.shape}")
print(f"Base value (expected value, log-odds): {shap_values.base_values[0]:.4f}")


## 6. SHAP Summary Plot (Beeswarm) — Global Feature Importance

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title("SHAP Summary Plot — Global Feature Importance")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_summary_beeswarm.png", bbox_inches="tight")
plt.show()


## 7. SHAP Bar Plot — Mean |SHAP Value| (Top 20 Features)

In [ ]:
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title("Mean |SHAP Value| — Top 20 Features")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_bar_top20.png", bbox_inches="tight")
plt.show()


## 8. SHAP Waterfall Plot — Correctly Predicted Default Case

Explains a single applicant the model correctly flagged as a default risk, showing exactly which features pushed the prediction toward default.

In [ ]:
y_pred = model.predict(X_sample)
y_proba = model.predict_proba(X_sample)[:, 1]

default_correct_idx = np.where((y_sample.values == 1) & (y_pred == 1))[0]
if len(default_correct_idx) == 0:
    raise ValueError("No correctly predicted default cases found in this sample.")
idx_default = default_correct_idx[0]

print(f"Explaining correctly predicted DEFAULT case at sample index {idx_default}")
print(f"True label: {y_sample.iloc[idx_default]}, Predicted: {y_pred[idx_default]}, "
      f"Predicted probability of default: {y_proba[idx_default]:.4f}")

shap.plots.waterfall(shap_values[idx_default], show=False)
plt.title("SHAP Waterfall — Correctly Predicted Default Case")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_waterfall_default_case.png", bbox_inches="tight")
plt.show()


## 9. SHAP Waterfall Plot — Correctly Predicted Non-Default Case

In [ ]:
nondefault_correct_idx = np.where((y_sample.values == 0) & (y_pred == 0))[0]
if len(nondefault_correct_idx) == 0:
    raise ValueError("No correctly predicted non-default cases found in this sample.")
idx_nondefault = nondefault_correct_idx[0]

print(f"Explaining correctly predicted NON-DEFAULT case at sample index {idx_nondefault}")
print(f"True label: {y_sample.iloc[idx_nondefault]}, Predicted: {y_pred[idx_nondefault]}, "
      f"Predicted probability of default: {y_proba[idx_nondefault]:.4f}")

shap.plots.waterfall(shap_values[idx_nondefault], show=False)
plt.title("SHAP Waterfall — Correctly Predicted Non-Default Case")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_waterfall_nondefault_case.png", bbox_inches="tight")
plt.show()


## 10. SHAP Dependence Plot — EXT_SOURCE_3

`EXT_SOURCE_3` is typically the single most predictive feature in this dataset (a normalized external credit bureau score). This plot shows how the feature's value relates to its SHAP contribution, colored by the strongest interacting feature.

In [ ]:
shap.plots.scatter(shap_values[:, "EXT_SOURCE_3"], color=shap_values, show=False)
plt.title("SHAP Dependence Plot — EXT_SOURCE_3")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_dependence_ext_source_3.png", bbox_inches="tight")
plt.show()


## 11. Top 10 Most Important Features (Mean |SHAP Value|)

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": X_sample.columns,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

print("Top 10 most important features by mean |SHAP value|:")
print(importance_df.head(10).to_string(index=False))
